In [2]:
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity 

In [3]:
model=joblib.load('recommender.pkl')
tfidf=model['tfidf']
km=model['km']
cluster_role=model['cluster_roles']
jobs=model['jobs']
job_vecs=tfidf.transform(jobs['required_skills'])

In [16]:
def recommand(user_skills,n):
    A = tfidf.transform([user_skills])
    cluster = int(km.predict(A)[0])
    recommended_titles = cluster_role.get(cluster, [])
    result = jobs[jobs["job_title"].isin(recommended_titles)].copy()
    if result.empty:
        result = jobs.copy()
    result_vecs = tfidf.transform(result["required_skills"].fillna(""))
    result["similarity_score"] = cosine_similarity(A,result_vecs)[0]
    return result.sort_values("similarity_score",ascending=False).head(n)

In [5]:
def get_fit_score(user_skills, target_job_title):
    job_row = jobs[jobs["job_title"] == target_job_title]
    
    if job_row.empty:
        return 0.0  
        
    target_skills = job_row["required_skills"].values[0]
    

    user_vec = tfidf.transform([user_skills])
    job_vec = tfidf.transform([target_skills])
    
    score = float(cosine_similarity(user_vec, job_vec)[0][0])
    return round(max(0.0, min(1.0, score)), 4)

In [18]:
recommand("Python, Java, Node.js, SQL, Git, Docker",5)

,job_title,required_skills,similarity_score
4,Backend Developer Intern,"Python, Java, Node.js, SQL, Git, Docker",1.000000
7,Software Engineer Intern,"Python, Java, C++, SQL, Git, Docker",0.759854
2,Python Developer Intern,"Python, Django, Flask, FastAPI, SQL, Git",0.264675
6,Data Science Intern,"Python, SQL, Pandas, NumPy, Machine Learning, ...",0.150060
0,Machine Learning Intern,"Python, Machine Learning, Pandas, NumPy, Sciki...",0.130959


In [13]:
get_fit_score("Python, Numpy, Node.js, Pandas, Html,Css, Docker","Python Developer Intern")

0.0572